In [1]:
import pandas as pd
import matplotlib.pyplot as plt 
import numpy as np


In [23]:
# Loading Datasets 

# Augmented and cleaned chch airbnb_data
df = pd.read_csv("sa_aug_cleaned_chch_airbnb.csv")

# tenancy data
ts = pd.read_csv("../data/Detailed-Quarterly-Tenancy-Q1-2020-Q3-2026.csv")

In [9]:
df.head()

,Unnamed: 0,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,...,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365,number_of_reviews_ltm,license,year_month,quarter,SA22019_V1_00
0,0,204050,"wee brick bungalow, Christchurch NZ",976173.0,Ian,Christchurch City,Fendalton Ward,-43.51148,172.59658,Private room,...,8,2014-04-14,0.05,1.0,365,0,NaN,2025-10-01,2025Q4,320800
1,1,204050,"wee brick bungalow, Christchurch NZ",976173.0,Ian,Christchurch City,Fendalton Ward,-43.51148,172.59658,Private room,...,8,2014-04-14,0.05,1.0,364,0,NaN,2025-11-01,2025Q4,320800
2,2,319770,Gateway to the South Island,1639584.0,Jennifer,Christchurch City,Innes Ward,-43.50961,172.65552,Private room,...,3,2015-01-09,0.02,1.0,364,0,NaN,2025-10-01,2025Q4,325900
3,3,319770,Gateway to the South Island,1639584.0,Jennifer,Christchurch City,Innes Ward,-43.50961,172.65552,Private room,...,3,2015-01-09,0.02,1.0,363,0,NaN,2025-11-01,2025Q4,325900
4,4,319770,Gateway to the South Island,1639584.0,Jennifer,Christchurch City,Innes Ward,-43.50961,172.65552,Private room,...,3,2015-01-09,0.02,1.0,364,0,NaN,2025-12-01,2025Q4,325900


In [10]:
df_group = df.groupby(["id", "quarter"])
df = df.drop(columns=["Unnamed: 0"], errors = "ignore")

df["year_month"] = pd.to_datetime(df["year_month"])

df_group = df.groupby(["id", "quarter"])


df_quarterly = df_group.agg(
    year_month = ("year_month", "first"),
    price = ("price", "mean"),
    number_of_reviews = ("number_of_reviews", "max"),
    minimum_nights = ("minimum_nights", "mean"), 
    host_id = ("host_id", "first"),
    host_name = ("host_name", "first"),
    latitude = ("latitude", "first"),
    longitude = ("longitude", "first"), 
    room_type = ("room_type", "first"),
    SA22019_V1_00 = ("SA22019_V1_00", "first")

).reset_index()

In [42]:
df_quarterly["quarter"] = pd.PeriodIndex(
    df_quarterly["quarter"],
    freq = "Q")
df_quarterly.head()

,id,quarter,year_month,price,number_of_reviews,minimum_nights,host_id,host_name,latitude,longitude,room_type,SA22019_V1_00
0,204050,2025Q4,2025-10-01,96.5,8,13.0,976173.0,Ian,-43.51148,172.59658,Private room,320800
1,319770,2025Q4,2025-10-01,112.0,3,1.0,1639584.0,Jennifer,-43.50961,172.65552,Private room,325900
2,319770,2026Q1,2026-01-01,198.0,3,1.0,1639584.0,Jennifer,-43.50961,172.65552,Private room,325900
3,319770,2026Q2,2026-04-01,198.0,3,1.0,1639584.0,Jennifer,-43.50961,172.65552,Private room,325900
4,737452,2025Q4,2025-10-01,330.0,99,2.0,3851282.0,Susan Margaret,-43.82343,172.94643,Entire home/apt,333300


In [45]:
df_quarterly["quarter"].dtype

period[Q-DEC]

In [46]:
ts.head()

,TimeFrame,Location Id,Dwelling Type,Number Of Beds,Total Bonds,Active Bonds,Closed Bonds,Median Rent,Geometric Mean Rent,Upper Quartile Rent,Lower Quartile Rent,Log Std Dev Weekly Rent,quarter
0,2026-04-01,NaN,ALL,1,2064,19563,1176,NaN,NaN,NaN,NaN,NaN,2026Q2
1,2026-04-01,-99.0,ALL,1,22014,113820,6969,550.0,522.0,660.0,450.0,0.4144,2026Q2
2,2026-04-01,100100.0,ALL,1,6,9,0,425.0,408.0,475.0,370.0,0.2834,2026Q2
3,2026-04-01,100600.0,ALL,1,6,12,3,588.0,585.0,720.0,545.0,0.2550,2026Q2
4,2026-04-01,100700.0,ALL,1,6,30,3,550.0,530.0,579.0,473.0,0.1588,2026Q2


In [47]:
# converting timeframe to same quarterly format
ts["TimeFrame"] = pd.to_datetime(ts["TimeFrame"])
ts["quarter"] = ts ["TimeFrame"].dt.to_period("Q")


ts[["TimeFrame", "quarter"]].head()

,TimeFrame,quarter
0,2026-04-01,2026Q2
1,2026-04-01,2026Q2
2,2026-04-01,2026Q2
3,2026-04-01,2026Q2
4,2026-04-01,2026Q2


In [48]:
ts["Location Id"].dropna().head(20)

1        -99.0
2     100100.0
3     100600.0
4     100700.0
5     100800.0
6     101100.0
7     102800.0
8     103000.0
9     103100.0
10    103500.0
11    103900.0
12    105500.0
13    105800.0
14    105900.0
15    106100.0
16    106200.0
17    106500.0
18    106600.0
19    106800.0
20    106900.0
Name: Location Id, dtype: float64

In [49]:

# Verifying that the codes overlap 
airbnb_codes = set(df_quarterly["SA22019_V1_00"].dropna().astype(int))
tenancy_codes = set(ts["Location Id"].dropna().astype(int))

len(airbnb_codes.intersection(tenancy_codes))

155

In [50]:
ts.groupby(["Location Id", "quarter"]).size().value_counts()

4     10109
2      6985
6      5596
5      3437
8      2743
7      2532
1      2248
3      2006
9      1505
10     1337
11      838
12      690
13      438
14      378
15      249
16      180
17      127
18      103
19       62
20       43
21       37
22       22
25       15
24       12
23       11
26        8
28        5
30        5
41        5
27        4
29        4
37        3
42        3
46        3
40        3
39        3
36        2
31        2
47        1
50        1
54        1
51        1
35        1
33        1
dtype: int64

In [51]:
ts[
    (ts["Location Id"] == 320800) &
    (ts["quarter"] == pd.Period("2025Q4"))
]

,TimeFrame,Location Id,Dwelling Type,Number Of Beds,Total Bonds,Active Bonds,Closed Bonds,Median Rent,Geometric Mean Rent,Upper Quartile Rent,Lower Quartile Rent,Log Std Dev Weekly Rent,quarter
21568,2025-10-01,320800.0,ALL,ALL,15,261,9,570.0,618.0,688.0,545.0,0.1999,2025Q4
26951,2025-10-01,320800.0,House,ALL,12,210,6,580.0,623.0,691.0,542.0,0.2055,2025Q4


In [52]:
# Filtering the tenency dataset to dwelling type = "ALL"
ts_all = ts[
    (ts["Dwelling Type"] == "ALL") &
    (ts["Number Of Beds"] == "ALL")].copy()
    

ts_all.groupby(["Location Id", "quarter"]).size().value_counts()

1    40884
dtype: int64

In [56]:
# Matching area-codes
ts_all["Location Id"] = ts_all["Location Id"].astype("Int64")
ts_all["Location Id"].dtype
ts_all = ts_all.dropna(subset=["Location Id"])

In [57]:
# Joining the tenancy and quarterly datasets
airbnb_tenancy_joined = df_quarterly.merge(
    ts_all,
    left_on = ["SA22019_V1_00", "quarter"], 
    right_on = ["Location Id", "quarter"], 
    how = "left"
)

In [59]:
# checking the join 
airbnb_tenancy_joined["Median Rent"].isna().value_counts()

False    8861
True     1494
Name: Median Rent, dtype: int64

In [60]:
# checking if unmatched observations are biased toward one quarter 
airbnb_tenancy_joined[
    airbnb_tenancy_joined["Median Rent"].isna()
]["quarter"].value_counts().sort_index()

2025Q4    556
2026Q1    434
2026Q2    504
Freq: Q-DEC, Name: quarter, dtype: int64

In [61]:
# Seeing how many unmatched 
unmatched = airbnb_tenancy_joined[
    airbnb_tenancy_joined["Median Rent"].isna()
]

unmatched["SA22019_V1_00"].nunique()

46

In [69]:
# reading to csv
# Exporting the joined dataset (previously this exported the monthly df by mistake)
airbnb_tenancy_joined.to_csv("airbnb_tenancy_joined.csv", index=False)